## Scraper de RaceResult España (my.raceresult.com)

RaceResult es una plataforma de cronometraje internacional muy usada también en
España. A diferencia de Kronoak, aquí **no hace falta ni BeautifulSoup ni HTML**:
el listado de eventos (`https://my.raceresult.com/events/`) se rellena en el propio
navegador llamando a una API JSON pública:

```
GET https://my.raceresult.com/RREvents/list?country=724&modes=topResults,topUpcoming&limit=...
```

`country=724` es el código numérico ISO 3166-1 de España (lo hemos sacado
inspeccionando la petición que dispara el propio buscador de la web al filtrar
por "Spain"). Con un `limit` suficientemente alto no hace falta paginar como en
Kronoak: se piden dos "modos" (`topResults` y `topUpcoming`, que entre los dos
cubren histórico + próximas carreras — lo hemos comprobado: `topUpcoming` añade
carreras que no salen en `topResults`) y con eso ya sale el catálogo completo.

**¿Por qué solo ~760 carreras españolas, si otras fuentes del proyecto tienen
muchas más filas?** Lo hemos comprobado a fondo porque al principio también nos
dio la sensación de que era poco: hemos probado todos los "modos" que usa el
propio buscador de la web (`topResults`, `topUpcoming`, `last`, `next`, `map`,
e incluso modos inventados que el servidor no reconoce y que caen a un
comportamiento por defecto) y **todos convergen en el mismo conjunto**, entre
759 y 761 eventos (los 2 de más son registros con fecha basura tipo
`0001-01-01`, claramente residuales). No hay paginación oculta ni un modo
"archivo completo" más grande. La explicación más plausible es simplemente que
RaceResult es un software de cronometraje internacional con poca cuota de
mercado en España comparado con plataformas específicamente españolas como las
de otras fuentes de este proyecto (mychip, ccnorte, sportmaniacs...) — no es un
fallo de nuestro scraper, es el tamaño real del catálogo de RaceResult aquí.
Por fechas, el catálogo va de 2001 a eventos ya programados para 2027.

Cada carrera trae ya nombre, fechas, tipo de deporte, distancias, ubicación **y
coordenadas (lat/lon)** — nos ahorramos la geocodificación por nombre que
tuvimos que hacer a mano en xipgroc, aquí podemos geocodificar al revés
(coordenadas -> municipio/provincia), que es más fiable.

### Clasificaciones por corredor: sí son automatizables

En un primer vistazo pensamos que los resultados individuales (la tabla de
corredores con tiempos) no se podían automatizar sin un navegador, porque la
tabla la carga un widget (`RRPublish`) y parecía necesitar un token de sesión
privado. Investigando más a fondo **sí hay manera**, y sin necesidad de
navegador:

1. Cada evento expone un endpoint público de configuración:
   `https://my.raceresult.com/{id}/results/config?lang=es&sanitize=true`.
   Devuelve, sin login ni cabeceras especiales, una `key` pública (el "token"
   que nos preocupaba, pero es de solo lectura y accesible para cualquiera),
   el servidor concreto que aloja los datos de ese evento en concreto (no
   siempre `my.raceresult.com`: puede ser `my1`, `my2`, `my4`...) y la lista
   de "listas de resultados" que ha configurado el organizador (normalmente
   una por distancia/categoría).
2. Con esa `key` se pide cada lista al servidor indicado:
   `https://{server}/{id}/results/list?key={key}&listname={nombre}&page=results&contest={id}&r=all&l=0&openedGroups={}&term=`.
   La respuesta es JSON con los datos de todos los corredores de esa lista
   **en una sola petición** (`r=all`), no hace falta paginar corredor a
   corredor. El filtro de "Género" que se ve en la web (Overall/Femenino/
   Masculino) es un filtro de la propia página sobre estos mismos datos, no
   una petición aparte — ya vienen todos juntos.

Cada organizador diseña sus listas a su manera (columnas, idioma, si agrupan
por sexo/categoría o no...) — no hay un esquema común entre eventos, y la
forma de los datos también varía: a veces es una lista plana de corredores, a
veces viene ya agrupada por categoría/sexo (p. ej. "Hombres"/"Mujeres" como
grupos separados dentro de la misma respuesta). Por eso, igual que con el
resto de fuentes del proyecto (mychip, ccnorte, sportmaniacs...), guardamos
las clasificaciones **en crudo**, sin forzar un esquema común: cada fila lleva
el id de evento, el nombre de la lista, el grupo (si lo hay) y todas las
columnas que haya definido ese organizador.

In [1]:
import ssl
import csv
import json
import time
import urllib.request
import urllib.parse
from pathlib import Path

import pandas as pd

# ==============================================================================
# 1. CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
# Igual que en los scrapers de xipgroc/Kronoak: desactivamos la verificación
# del certificado SSL por si el entorno no tiene los certificados raíz al día.
context = ssl._create_unverified_context()

CODIGO_PAIS_ESPANA = 724  # código numérico ISO 3166-1 que usa RaceResult para España
URL_LISTADO = "https://my.raceresult.com/RREvents/list"
PARAMS_BASE = {
    'country': CODIGO_PAIS_ESPANA,
    'group': 0, 'user': 0, 'userID': 0,
    'geoLocation': 'IP', 'lang': 'es',
}


def obtener_json(url, timeout=20):
    """Descarga una URL y la interpreta como JSON, o devuelve None si falla."""
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, context=context, timeout=timeout) as resp:
            return json.loads(resp.read().decode('utf-8'))
    except Exception as e:
        print(f"  🚨 Error al descargar {url}: {e}")
        return None


def descargar_eventos(modo, limite=5000):
    """Pide un 'modo' de la API de RaceResult (topResults / topUpcoming) con
    un límite alto para traer todo de una vez. Avisa si aun así quedara más
    (HasMore=True), por si en el futuro hay más de `limite` carreras."""
    params = dict(PARAMS_BASE)
    params['modes'] = modo
    params['limit'] = limite
    url = URL_LISTADO + '?' + urllib.parse.urlencode(params)

    data = obtener_json(url)
    if not data:
        return []

    bloque = data[0]
    print(f"  Modo '{modo}': {len(bloque['Events'])} carreras (HasMore={bloque['HasMore']})")
    if bloque['HasMore']:
        print(f"  ⚠️ HasMore=True con limit={limite}; sube 'limite' si quieres "
              f"asegurarte de tener el listado completo.")
    return bloque['Events']

### Descarga del catálogo de carreras españolas

Combinamos `topResults` (histórico + populares) y `topUpcoming` (próximas
carreras) porque, como hemos comprobado antes de escribir este notebook, no son
subconjuntos el uno del otro — cada modo trae carreras que el otro no trae — y
luego eliminamos duplicados por `id` de evento.

In [2]:
# ==============================================================================
# 2. DESCARGA DEL CATÁLOGO (sin necesidad de paginar página a página)
# ==============================================================================
print("Descargando el catálogo de carreras españolas de RaceResult...")

eventos_raw = descargar_eventos('topResults') + descargar_eventos('topUpcoming')

df_raceresult = pd.DataFrame(eventos_raw).drop_duplicates(subset='id').reset_index(drop=True)
print(f"\nTotal de carreras españolas únicas: {len(df_raceresult)}")

# Nos quedamos con las columnas útiles y las renombramos siguiendo el mismo
# esquema en español que usamos en el resto de fuentes del proyecto.
df_raceresult = df_raceresult.rename(columns={
    'name': 'nombre_carrera',
    'dateFrom': 'fecha_inicio',
    'dateTo': 'fecha_fin',
    'eventTypeName': 'tipo_deporte',
    'distances': 'distancias',
    'location': 'ubicacion',
    'region': 'region',
    'countryName': 'pais',
    'countryCode': 'codigo_pais',
    'lat': 'lat',
    'lng': 'lon',
})[[
    'id', 'nombre_carrera', 'fecha_inicio', 'fecha_fin', 'tipo_deporte',
    'distancias', 'ubicacion', 'region', 'pais', 'codigo_pais', 'lat', 'lon',
]]

df_raceresult['fecha_inicio'] = pd.to_datetime(df_raceresult['fecha_inicio'])
df_raceresult['fecha_fin'] = pd.to_datetime(df_raceresult['fecha_fin'])
df_raceresult['url_evento'] = 'https://my.raceresult.com/' + df_raceresult['id'].astype(str) + '/'

print()
print("Carreras por tipo de deporte:")
print(df_raceresult['tipo_deporte'].value_counts())
df_raceresult.head()

Descargando el catálogo de carreras españolas de RaceResult...
  Modo 'topResults': 738 carreras (HasMore=False)
  Modo 'topUpcoming': 24 carreras (HasMore=False)

Total de carreras españolas únicas: 762

Carreras por tipo de deporte:
tipo_deporte
Running                  203
Bicicleta de montaña     130
Carreras de montaña      107
Otros                    104
Carrera de Obstáculos     57
Ciclismo                  57
Triatlón                  30
Natación                  26
Cicloturismo              17
Carrera de fitness        12
Deportes motor             5
BMX                        3
Acuatlón                   3
Ciclocross                 3
Snowboard                  1
Marcha atlética            1
Motocross                  1
Atletismo                  1
Duatlón                    1
Name: count, dtype: int64


,id,nombre_carrera,fecha_inicio,fecha_fin,tipo_deporte,distancias,ubicacion,region,pais,codigo_pais,lat,lon,url_evento
0,422137,XIII DHI Puntallana 2026,2026-09-13,2026-09-13,Bicicleta de montaña,,Puntallana,,España,ES,28.3914,-16.5221,https://my.raceresult.com/422137/
1,422376,II MINIDHU CONCELLO DE OURENSE,2026-09-13,2026-09-13,Bicicleta de montaña,,OURENSE,,España,ES,42.3401,-7.8261,https://my.raceresult.com/422376/
2,422738,Titán Race,2026-09-12,2026-09-12,Carrera de fitness,,Puerto Real,,España,ES,36.5282,-6.1914,https://my.raceresult.com/422738/
3,419752,TRIPLE CORONA ILLAS ATLÁNTICAS 2026,2026-09-12,2026-09-12,Natación,,Baiona,,España,ES,42.1192,-8.8478,https://my.raceresult.com/419752/
4,386422,XI Enduro Puntallana Flow Trails 2026 - Campeo...,2026-09-12,2026-09-12,Bicicleta de montaña,,Puntallana,,España,ES,28.6964,-17.7683,https://my.raceresult.com/386422/


### Geocodificación inversa (lat/lon -> municipio/provincia)

En xipgroc tuvimos que *adivinar* el lugar a partir del nombre de la carrera y
geocodificarlo (best-effort, con bastantes fallos). Aquí RaceResult ya nos da
coordenadas reales por evento, así que hacemos el camino inverso con
Nominatim/OpenStreetMap — mucho más fiable — y con el mismo patrón de
checkpoint reanudable que usamos en xipgroc (`raceresult_ubicaciones.csv`).

Para no repetir la misma consulta cuando varias carreras comparten sede (muy
habitual: la misma prueba en distintos años, o varias distancias del mismo
evento), redondeamos las coordenadas a 4 decimales (~11 m de precisión) y
geocodificamos solo las combinaciones únicas.

Requiere `pip install geopy`. Nominatim limita a 1 petición/segundo. Pon
`REALIZAR_GEOCODIFICACION_INVERSA = False` si de momento solo quieres el
catálogo sin ubicación administrativa.

In [3]:
# ==============================================================================
# 3. (OPCIONAL) GEOCODIFICACIÓN INVERSA CON NOMINATIM
# ==============================================================================
REALIZAR_GEOCODIFICACION_INVERSA = True

try:
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError
    GEOPY_OK = True
except ImportError:
    GEOPY_OK = False
    print("geopy no está instalado. Ejecuta `pip install geopy` si quieres "
          "obtener municipio/provincia a partir de las coordenadas. Esta "
          "parte es opcional.")


def geocodificar_inverso(df, out_dir, pausa_segundos: float = 1.1):
    out_path = Path(out_dir)
    csv_ubic = out_path / "raceresult_ubicaciones.csv"

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        # Importante: redondeamos lat/lon del checkpoint a float (no strings)
        # para que la clave de caché sea comparable con las claves calculadas
        # más abajo a partir de df. Comparar strings tal cual ("29.0720" vs
        # "29.072") rompía la reanudación y hacía re-geocodificar todo.
        prev['lat'] = prev['lat'].astype(float).round(4)
        prev['lon'] = prev['lon'].astype(float).round(4)
        cache = {(row['lat'], row['lon']): row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} coordenadas ya geocodificadas")

    geolocator = Nominatim(user_agent="raceresult_scraper_claudia")

    # Coordenadas únicas (redondeadas) y válidas -- (0, 0) es "sin ubicación",
    # no un punto real, así que las dejamos fuera.
    coords_validas = df.loc[(df['lat'] != 0) | (df['lon'] != 0), ['lat', 'lon']].round(4)
    claves_unicas = list(coords_validas.drop_duplicates().itertuples(index=False, name=None))
    pendientes = [c for c in claves_unicas if c not in cache]
    print(f"Coordenadas a geocodificar: {len(pendientes)} (de {len(claves_unicas)} únicas)")

    campos = ['lat', 'lon', 'municipio', 'comarca', 'provincia']
    write_header = not csv_ubic.exists()
    with open(csv_ubic, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, (lat, lon) in enumerate(pendientes, 1):
            fila = {c: None for c in campos}
            fila['lat'], fila['lon'] = lat, lon
            try:
                loc = geolocator.reverse(
                    (lat, lon), exactly_one=True, language='es',
                    addressdetails=True, timeout=10,
                )
                if loc:
                    addr = loc.raw.get('address', {})
                    fila['municipio'] = (
                        addr.get('city') or addr.get('town') or addr.get('village')
                        or addr.get('municipality')
                    )
                    fila['comarca'] = addr.get('county')
                    fila['provincia'] = addr.get('province') or addr.get('state')
            except GeopyError as e:
                print(f"  [{lat}, {lon}] ERROR de geocodificación: {e}")
            except Exception as e:
                print(f"  [{lat}, {lon}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[(lat, lon)] = fila

            if i % 25 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    df_ubic = pd.DataFrame(cache.values())
    print(f"Geocodificadas con éxito: {df_ubic['municipio'].notna().sum()} de {len(df_ubic)}")
    return df_ubic


if REALIZAR_GEOCODIFICACION_INVERSA and GEOPY_OK:
    # Misma convención de carpetas que el resto de fuentes del proyecto:
    # notebooks/scraping/<este notebook>.ipynb -> ../../data/raw/raceresult/
    OUT_DIR = Path("../../data/raw/raceresult")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    df_raceresult['lat_r'] = df_raceresult['lat'].round(4)
    df_raceresult['lon_r'] = df_raceresult['lon'].round(4)

    df_ubicaciones = geocodificar_inverso(df_raceresult, out_dir=OUT_DIR)
    df_ubicaciones['lat_r'] = df_ubicaciones['lat'].astype(float).round(4)
    df_ubicaciones['lon_r'] = df_ubicaciones['lon'].astype(float).round(4)

    df_raceresult = df_raceresult.merge(
        df_ubicaciones[['lat_r', 'lon_r', 'municipio', 'comarca', 'provincia']],
        on=['lat_r', 'lon_r'], how='left',
    ).drop(columns=['lat_r', 'lon_r'])
else:
    df_raceresult['municipio'] = None
    df_raceresult['comarca'] = None
    df_raceresult['provincia'] = None

df_raceresult.head()

Checkpoint: 343 coordenadas ya geocodificadas
Coordenadas a geocodificar: 2 (de 345 únicas)
  [42.3401, -7.8261] ERROR de geocodificación: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /reverse?lat=42.3401&lon=-7.8261&format=json&accept-language=es&addressdetails=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)')))
  [36.5282, -6.1914] ERROR de geocodificación: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /reverse?lat=36.5282&lon=-6.1914&format=json&accept-language=es&addressdetails=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)')))
  Progreso: 2/2
CSV de ubicaciones: /Users/claudiarm2002/Desktop/TFM/data/raw/raceresult/raceresult_ubicaciones.csv
Geocodif

,id,nombre_carrera,fecha_inicio,fecha_fin,tipo_deporte,distancias,ubicacion,region,pais,codigo_pais,lat,lon,url_evento,municipio,comarca,provincia
0,422137,XIII DHI Puntallana 2026,2026-09-13,2026-09-13,Bicicleta de montaña,,Puntallana,,España,ES,28.3914,-16.5221,https://my.raceresult.com/422137/,La Orotava,NaN,Santa Cruz de Tenerife
1,422376,II MINIDHU CONCELLO DE OURENSE,2026-09-13,2026-09-13,Bicicleta de montaña,,OURENSE,,España,ES,42.3401,-7.8261,https://my.raceresult.com/422376/,None,None,None
2,422738,Titán Race,2026-09-12,2026-09-12,Carrera de fitness,,Puerto Real,,España,ES,36.5282,-6.1914,https://my.raceresult.com/422738/,None,None,None
3,419752,TRIPLE CORONA ILLAS ATLÁNTICAS 2026,2026-09-12,2026-09-12,Natación,,Baiona,,España,ES,42.1192,-8.8478,https://my.raceresult.com/419752/,Baiona,Vigo,Pontevedra
4,386422,XI Enduro Puntallana Flow Trails 2026 - Campeo...,2026-09-12,2026-09-12,Bicicleta de montaña,,Puntallana,,España,ES,28.6964,-17.7683,https://my.raceresult.com/386422/,Santa Cruz de La Palma,NaN,Santa Cruz de Tenerife


In [4]:
# Vista previa del catálogo con ubicación ya cruzada (la exportación final,
# con los finishers por modalidad ya agregados, va al final del notebook,
# después de las clasificaciones).
print(df_raceresult.shape)
df_raceresult.head()

(762, 16)


,id,nombre_carrera,fecha_inicio,fecha_fin,tipo_deporte,distancias,ubicacion,region,pais,codigo_pais,lat,lon,url_evento,municipio,comarca,provincia
0,422137,XIII DHI Puntallana 2026,2026-09-13,2026-09-13,Bicicleta de montaña,,Puntallana,,España,ES,28.3914,-16.5221,https://my.raceresult.com/422137/,La Orotava,NaN,Santa Cruz de Tenerife
1,422376,II MINIDHU CONCELLO DE OURENSE,2026-09-13,2026-09-13,Bicicleta de montaña,,OURENSE,,España,ES,42.3401,-7.8261,https://my.raceresult.com/422376/,None,None,None
2,422738,Titán Race,2026-09-12,2026-09-12,Carrera de fitness,,Puerto Real,,España,ES,36.5282,-6.1914,https://my.raceresult.com/422738/,None,None,None
3,419752,TRIPLE CORONA ILLAS ATLÁNTICAS 2026,2026-09-12,2026-09-12,Natación,,Baiona,,España,ES,42.1192,-8.8478,https://my.raceresult.com/419752/,Baiona,Vigo,Pontevedra
4,386422,XI Enduro Puntallana Flow Trails 2026 - Campeo...,2026-09-12,2026-09-12,Bicicleta de montaña,,Puntallana,,España,ES,28.6964,-17.7683,https://my.raceresult.com/386422/,Santa Cruz de La Palma,NaN,Santa Cruz de Tenerife


### Clasificaciones por corredor (opcional, tarda mucho más)

Descargamos, para cada carrera ya finalizada (`EventOver`), la configuración
de resultados y cada una de sus listas, siguiendo el mecanismo explicado en la
introducción. Como son ~760 carreras y cada una puede tener varias listas, son
varios miles de peticiones en total -- igual que con la geocodificación, el
proceso **guarda un checkpoint cada pocas carreras** (`raceresult_clasificaciones.csv`
+ dos ficheros de control) para poder interrumpirlo y reanudarlo sin perder lo
ya descargado ni repetir trabajo.

No forzamos un esquema común entre carreras (cada organizador nombra sus
columnas a su manera, como comentábamos arriba): cada fila lleva `id_evento`,
`nombre_carrera`, `lista` (la distancia/categoría) y `grupo` (si esa lista
viene agrupada, p. ej. por sexo) como columnas fijas, y luego todas las
columnas que haya definido ese organizador -- si una carrera no tiene alguna
columna que sí tiene otra, sencillamente queda en blanco (`NaN`) en sus filas,
es el mismo criterio que ya usa el resto de fuentes del proyecto con esquemas
variables.

Pon `MAX_EVENTOS_CLASIFICACIONES` a un número pequeño (p. ej. 20) para hacer
una prueba rápida antes de lanzar el proceso completo; déjalo en `None` para
procesar todas las carreras (puede tardar bastante más de una sesión --
simplemente vuelve a ejecutar la celda más adelante y continuará donde lo
dejó).

In [5]:
# ==============================================================================
# 5. (OPCIONAL) CLASIFICACIONES POR CORREDOR, CARRERA A CARRERA
# ==============================================================================
REALIZAR_DESCARGA_CLASIFICACIONES = True
PAUSA_CLASIFICACIONES = 0.4       # segundos de cortesía entre peticiones
GUARDAR_CADA_N_EVENTOS = 20       # frecuencia de checkpoint
MAX_EVENTOS_CLASIFICACIONES = None  # pon un número pequeño para probar primero


def obtener_config_resultados(event_id):
    """Configuración pública de resultados de un evento: incluye la 'key' de
    acceso (de solo lectura, sin login), el servidor real que aloja los datos
    de ESE evento concreto, y las listas de resultados que definió el
    organizador."""
    url = f"https://my.raceresult.com/{event_id}/results/config?lang=es&sanitize=true"
    return obtener_json(url)


def descargar_lista_resultados(event_id, server, key, nombre_lista, contest):
    """Pide TODOS los corredores de una lista de resultados de una vez
    (r=all): el filtro de género/categoría de la web es un filtro sobre estos
    mismos datos, no hace falta pedirlo aparte."""
    params = {
        'key': key, 'listname': nombre_lista, 'page': 'results',
        'contest': contest, 'r': 'all', 'l': 0, 'openedGroups': '{}', 'term': '',
    }
    url = f"https://{server}/{event_id}/results/list?" + urllib.parse.urlencode(params)
    return obtener_json(url)


def normalizar_filas_lista(resultado, event_id, nombre_carrera, lista_info):
    """La forma de 'data' varía según cómo haya configurado el organizador la
    lista: a veces es una lista plana de filas, a veces viene ya agrupada en
    un diccionario {grupo: [filas...]} (p. ej. agrupado por sexo o categoría).
    Normalizamos ambos casos al mismo formato de filas sueltas."""
    filas = []
    if not isinstance(resultado, dict):
        return filas
    data = resultado.get('data')
    campos = resultado.get('DataFields', [])
    if not data:
        return filas
    grupos = data if isinstance(data, dict) else {None: data}
    etiqueta_lista = lista_info.get('ShowAs') or lista_info.get('Name', '')
    for grupo, registros in grupos.items():
        if not isinstance(registros, list):
            continue
        for valores in registros:
            fila = {
                'id_evento': event_id,
                'nombre_carrera': nombre_carrera,
                'lista': etiqueta_lista,
                'grupo': grupo,
            }
            fila.update(dict(zip(campos, valores)))
            filas.append(fila)
    return filas


def descargar_clasificaciones(df, out_dir, pausa_segundos=0.4, guardar_cada=20, max_eventos=None):
    out_path = Path(out_dir)
    csv_clasif = out_path / "raceresult_clasificaciones.csv"
    checkpoint_file = out_path / "raceresult_clasificaciones_checkpoint.json"
    sin_resultados_file = out_path / "raceresult_eventos_sin_resultados.json"

    filas = []
    if csv_clasif.exists():
        filas = pd.read_csv(csv_clasif, dtype=str).to_dict('records')
        print(f"Checkpoint: {len(filas)} filas de clasificaciones ya descargadas")

    procesados = set(json.loads(checkpoint_file.read_text())) if checkpoint_file.exists() else set()
    sin_resultados = set(json.loads(sin_resultados_file.read_text())) if sin_resultados_file.exists() else set()

    pendientes = [
        (int(row['id']), row['nombre_carrera'])
        for _, row in df.iterrows()
        if int(row['id']) not in procesados and int(row['id']) not in sin_resultados
    ]
    if max_eventos is not None:
        pendientes = pendientes[:max_eventos]
    print(f"Eventos pendientes de clasificaciones: {len(pendientes)} (de {len(df)} totales; "
          f"{len(procesados)} ya descargados, {len(sin_resultados)} sin resultados/futuros)")

    for i, (event_id, nombre) in enumerate(pendientes, 1):
        try:
            cfg = obtener_config_resultados(event_id)
            listas = ((cfg or {}).get('TabConfig') or {}).get('Lists') or []
            if not cfg or not cfg.get('EventOver') or not listas:
                sin_resultados.add(event_id)
            else:
                key, server = cfg['key'], cfg['server']
                alguna_fila = False
                todas_las_listas_ok = True
                for lista in listas:
                    resultado = descargar_lista_resultados(
                        event_id, server, key, lista['Name'], lista.get('Contest', 0),
                    )
                    if resultado is None:
                        # obtener_json ya ha impreso el error; no propaga la
                        # excepción, así que lo detectamos aquí explícitamente.
                        todas_las_listas_ok = False
                    nuevas_filas = normalizar_filas_lista(resultado, event_id, nombre, lista)
                    filas.extend(nuevas_filas)
                    alguna_fila = alguna_fila or bool(nuevas_filas)
                    time.sleep(pausa_segundos)
                if not alguna_fila:
                    sin_resultados.add(event_id)
                elif todas_las_listas_ok:
                    procesados.add(event_id)
                # Si alguna lista falló pero otras sí trajeron filas, NO
                # marcamos el evento ni como procesado ni como sin resultados:
                # así se reintenta entero en la siguiente ejecución en vez de
                # perder para siempre la lista que falló. Las filas de las
                # listas que sí funcionaron no se duplican: se deduplican al
                # guardar (`.drop_duplicates()` más abajo).
        except Exception as e:
            print(f"  [{event_id}] {nombre}: ERROR -- {e}")
            # Lo marcamos como 'sin resultados' para no reintentarlo en bucle en
            # cada ejecución; si el fallo era puntual (p. ej. un corte de red),
            # basta con borrar su id de ese fichero de control para reintentarlo.
            sin_resultados.add(event_id)

        if i % guardar_cada == 0 or i == len(pendientes):
            pd.DataFrame(filas).drop_duplicates().to_csv(csv_clasif, index=False, encoding="utf-8-sig")
            checkpoint_file.write_text(json.dumps(sorted(procesados)))
            sin_resultados_file.write_text(json.dumps(sorted(sin_resultados)))
            print(f"  Progreso: {i}/{len(pendientes)} eventos -- {len(filas)} filas guardadas")

    print(f"CSV de clasificaciones: {csv_clasif.resolve()}")
    print(f"Eventos con clasificaciones: {len(procesados)} | sin resultados/futuros: {len(sin_resultados)}")
    return pd.DataFrame(filas)


if REALIZAR_DESCARGA_CLASIFICACIONES:
    OUTPUT_DIR = Path("../../data/raw/raceresult")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_clasificaciones = descargar_clasificaciones(
        df_raceresult, out_dir=OUTPUT_DIR,
        pausa_segundos=PAUSA_CLASIFICACIONES,
        guardar_cada=GUARDAR_CADA_N_EVENTOS,
        max_eventos=MAX_EVENTOS_CLASIFICACIONES,
    )
    print(df_clasificaciones.shape)
    df_clasificaciones.head()

Eventos pendientes de clasificaciones: 72 (de 762 totales; 78 ya descargados, 612 sin resultados/futuros)
  🚨 Error al descargar https://my2.raceresult.com/419334/results/list?key=4e5cf966af72e6539c2afebe1270b4db&listname=4.RESULTADOS+3+MANGAS%7C4.2-Resultados+-+Scratch&page=results&contest=0&r=all&l=0&openedGroups=%7B%7D&term=: HTTP Error 404: Not Found
  Progreso: 20/72 eventos -- 2865 filas guardadas
  Progreso: 40/72 eventos -- 5315 filas guardadas
  Progreso: 60/72 eventos -- 23620 filas guardadas
  🚨 Error al descargar https://my.raceresult.com/422673/results/config?lang=es&sanitize=true: HTTP Error 404: Not Found
  Progreso: 72/72 eventos -- 32713 filas guardadas
CSV de clasificaciones: /Users/claudiarm2002/Desktop/TFM/data/raw/raceresult/raceresult_clasificaciones.csv
Eventos con clasificaciones: 146 | sin resultados/futuros: 615
(32713, 537)


### Agregación por modalidad: de corredor a corredor a una fila por evento+modalidad

`raceresult_clasificaciones.csv` tiene una fila por **corredor**, y dentro de un mismo evento la columna `lista` no es la modalidad/distancia — es el **tipo de vista** que configuró el organizador (general, por sexo, por categoría, tiempos de paso intermedios, hora de salida, podio de premios...). Muchas veces la misma persona aparece en varias listas del mismo evento (comprobado con el caso "PONTEVEDRA 4 PICOS BIKE": la lista "Clasificación Categorías" tenía exactamente 958+545 filas, la suma de las dos listas "Tiempo final" de sus dos modalidades reales — el mismo corredor contado dos veces bajo otro nombre de lista).

Para tener "una fila por modalidad" como el resto de fuentes del proyecto:

1. Identificamos, dentro de cada evento, qué lista(s) son de verdad una clasificación general/final de una carrera — no listas parciales, de paso, de salida o de premios — por palabras clave.
2. Si dos listas seleccionadas del mismo evento tienen **exactamente** el mismo número de filas, es casi seguro que son los mismos corredores con otro nombre (p.ej. "CLASIFICACIÓN FINAL" y "CLASIFICACIÓN FINAL COMPLETA", 149 filas cada una) — nos quedamos solo con una.
3. Contamos finishers totales, y repartimos por sexo cuando se puede: por la columna `grupo`/`SEX` de la propia lista, o si no, buscando una lista "por sexo" hermana del mismo evento con el mismo número exacto de filas.

**Es una heurística por palabras clave, no una solución perfecta.** Con más de 100 nombres de lista distintos en solo 147 eventos (cada organizador configura RaceResult a su manera: carreras por etapas, series de pista, duatlones con clasificación combinada, pruebas de buceo con modalidades genuinamente distintas...), habrá casos raros mal resueltos o con algo de doble conteo residual — se documenta como limitación conocida, igual que el resto de fuentes del proyecto con sus propios problemas de formato de origen. El reparto por sexo, en concreto, solo se consigue determinar de forma fiable para una minoría de las filas (la mayoría de listas generales no llevan el sexo como columna propia); el resto se queda con `finisher_total` pero `finisher_h`/`finisher_d` en `NaN`.

In [6]:
import re
import unicodedata


def _normalizar_texto(t):
    if pd.isna(t):
        return ""
    return unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode("ascii").lower().strip()


_EXCLUIR_LISTA = re.compile(
    r"paso|salida|premio|podio|podium|qualif|clasificatoria|heat|1/4|1/2\s*final|vuelta rapida|"
    r"tiempos (natacion|bici|carrera)|after day|hora de|comprobar|retirad"
)
_VISTA_PARCIAL = re.compile(r"sexo|categor|local")
_PRIORIDAD_LISTA = [
    re.compile(r"\bgeneral\b"), re.compile(r"\bfinal"), re.compile(r"\bresultados\b"),
    re.compile(r"scratch"), re.compile(r"clasificaci"), re.compile(r"listado"),
    re.compile(r"\boverall\b"), re.compile(r"\bindividual\b"), re.compile(r"\bopen\b"),
]
_POR_SEXO = re.compile(r"sexo")
_PATRON_HOMBRE = re.compile(r"masculin|hombre|\bmen\b|varon|\bmasc\b|\bhomes\b|^m$|^m[_ -]|[_ -]m$")
_PATRON_MUJER = re.compile(r"femenin|mujer|\bwomen\b|\bfem\b|\bdones\b|^f$|^f[_ -]|[_ -]f$|^w$")


def _clasificar_listas_evento(listas_unicas):
    """Filtra las listas de un evento y devuelve las que parecen ser una
    clasificación general/final real (ver celda de arriba)."""
    candidatas = [l for l in listas_unicas if not _EXCLUIR_LISTA.search(_normalizar_texto(l))]
    puras = [l for l in candidatas if not _VISTA_PARCIAL.search(_normalizar_texto(l))]
    con_prioridad = [l for l in puras if any(p.search(_normalizar_texto(l)) for p in _PRIORIDAD_LISTA)]
    if con_prioridad:
        return con_prioridad
    if puras:
        return puras
    return candidatas  # solo quedan vistas parciales (por sexo/categoria/local) -- mejor eso que nada


def _quitar_duplicados_por_conteo(seleccionadas, conteos):
    """Si dos listas seleccionadas tienen el mismo nº de filas, nos quedamos
    solo con la de nombre más corto (más probable que sea la canónica)."""
    por_conteo = {}
    for lista in seleccionadas:
        por_conteo.setdefault(conteos[lista], []).append(lista)
    return [min(grupo, key=len) for grupo in por_conteo.values()]


def _detectar_sexo_fila(grupo, sexo_col):
    g = _normalizar_texto(grupo)
    if _PATRON_MUJER.search(g):
        return "D"
    if _PATRON_HOMBRE.search(g):
        return "H"
    s = _normalizar_texto(sexo_col)
    if s in ("f", "femenino", "female", "mujer"):
        return "D"
    if s in ("m", "masculino", "male", "hombre"):
        return "H"
    return None


RUTA_CLASIF = Path("../../data/raw/raceresult/raceresult_clasificaciones.csv")
clasif = pd.read_csv(RUTA_CLASIF, dtype=str)
clasif["_sexo"] = clasif.apply(
    lambda r: _detectar_sexo_fila(r.get("grupo"), r.get("SEX") if "SEX" in clasif.columns else None), axis=1
)

filas_agregadas = []
for event_id, grupo_evento in clasif.groupby("id_evento"):
    conteos = grupo_evento["lista"].value_counts().to_dict()
    seleccionadas = _clasificar_listas_evento(list(conteos.keys()))
    seleccionadas = _quitar_duplicados_por_conteo(seleccionadas, conteos)
    listas_por_sexo = [l for l in conteos if _POR_SEXO.search(_normalizar_texto(l))]

    for lista_sel in seleccionadas:
        sub = grupo_evento[grupo_evento["lista"] == lista_sel]
        n_total = len(sub)
        n_h = (sub["_sexo"] == "H").sum()
        n_d = (sub["_sexo"] == "D").sum()
        finisher_h, finisher_d = (n_h, n_d) if n_h + n_d == n_total and n_total > 0 else (None, None)

        # si esta lista no trae sexo, probamos con una lista "por sexo" hermana
        # que tenga EXACTAMENTE el mismo nº de corredores (misma carrera).
        if finisher_h is None:
            for lsx in listas_por_sexo:
                sub_sexo = grupo_evento[grupo_evento["lista"] == lsx]
                if len(sub_sexo) != n_total:
                    continue
                n_h2 = (sub_sexo["_sexo"] == "H").sum()
                n_d2 = (sub_sexo["_sexo"] == "D").sum()
                if n_h2 + n_d2 == n_total:
                    finisher_h, finisher_d = n_h2, n_d2
                    break

        filas_agregadas.append({
            "id": int(event_id), "modalidad_raceresult": lista_sel,
            "finisher_h": finisher_h, "finisher_d": finisher_d, "finisher_total": n_total,
        })

df_modalidades = pd.DataFrame(filas_agregadas)
print(f"Filas evento x modalidad: {len(df_modalidades)} (de {clasif['id_evento'].nunique()} eventos con clasificaciones)")
print(f"Con reparto por sexo determinado: {df_modalidades['finisher_h'].notna().sum()} de {len(df_modalidades)} "
      f"({100 * df_modalidades['finisher_h'].notna().mean():.1f}%) -- el resto se queda solo con finisher_total, "
      "sin dato fiable de sexo en origen.")
df_modalidades.head(10)

Filas evento x modalidad: 89 (de 69 eventos con clasificaciones)
Con reparto por sexo determinado: 21 de 89 (23.6%) -- el resto se queda solo con finisher_total, sin dato fiable de sexo en origen.


,id,modalidad_raceresult,finisher_h,finisher_d,finisher_total
0,101182,Clasificación general,222.0,104.0,326
1,101238,Clasificación general,NaN,NaN,258
2,101238,Clasificación Relevos|Cls. Modalidad,NaN,NaN,41
3,101238,Clasificación por modalidad,NaN,NaN,9
4,103015,Clasificación general,133.0,49.0,182
5,103018,Clasificación general,173.0,133.0,306
6,103021,Clasificación general,193.0,141.0,334
7,103022,Clasificación general,140.0,112.0,252
8,103022,Clasificación Club,NaN,NaN,1
9,103023,Clasificación general,137.0,111.0,248


### Exportación final: catálogo + modalidades con finishers

Antes se exportaba solo el catálogo de eventos (una fila por evento, sin finishers). Ahora se exporta una fila por evento+modalidad, con los finishers ya contados — mismo nivel de detalle que el resto de fuentes del proyecto. Los eventos sin clasificaciones descargadas (ver `raceresult_eventos_sin_resultados.json`, sobre todo carreras futuras o sin resultados publicados) quedan fuera de este fichero: no tiene sentido "finishers por modalidad" para un evento sin resultados.

In [7]:
df_raceresult_final = df_modalidades.merge(df_raceresult, on="id", how="left")

OUTPUT_DIR = Path("../../data/raw/raceresult")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ruta_salida = OUTPUT_DIR / "DF_RACERESULT_SUCIO.csv"

try:
    df_raceresult_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"Archivo guardado correctamente en: {ruta_salida.resolve()}")
except Exception as e:
    df_raceresult_final.to_csv("DF_RACERESULT_SUCIO_backup.csv", index=False, encoding="utf-8-sig")
    print("No se ha podido guardar en la carpeta de datos; se ha guardado como "
          "'DF_RACERESULT_SUCIO_backup.csv' en el directorio actual.")

print(df_raceresult_final.shape)
df_raceresult_final.head()

Archivo guardado correctamente en: /Users/claudiarm2002/Desktop/TFM/data/raw/raceresult/DF_RACERESULT_SUCIO.csv
(89, 20)


,id,modalidad_raceresult,finisher_h,finisher_d,finisher_total,nombre_carrera,fecha_inicio,fecha_fin,tipo_deporte,distancias,ubicacion,region,pais,codigo_pais,lat,lon,url_evento,municipio,comarca,provincia
0,101182,Clasificación general,222.0,104.0,326,Trail la Vegueta 2018,2018-07-07,2018-07-07,Running,"5K,10K",Tinajo,,España,ES,29.0667,-13.6771,https://my.raceresult.com/101182/,Tinajo,NaN,Las Palmas
1,101238,Clasificación general,NaN,NaN,258,Relevos el Cuchillo 2018,2018-11-10,2018-11-10,Running,,El Cuchillo,,España,ES,29.0814,-13.6630,https://my.raceresult.com/101238/,Tinajo,NaN,Las Palmas
2,101238,Clasificación Relevos|Cls. Modalidad,NaN,NaN,41,Relevos el Cuchillo 2018,2018-11-10,2018-11-10,Running,,El Cuchillo,,España,ES,29.0814,-13.6630,https://my.raceresult.com/101238/,Tinajo,NaN,Las Palmas
3,101238,Clasificación por modalidad,NaN,NaN,9,Relevos el Cuchillo 2018,2018-11-10,2018-11-10,Running,,El Cuchillo,,España,ES,29.0814,-13.6630,https://my.raceresult.com/101238/,Tinajo,NaN,Las Palmas
4,103015,Clasificación general,133.0,49.0,182,Carrera Popular EL Quíquere 2018,2018-08-04,2018-08-04,Running,"5K,10K",Tias,,España,ES,28.9242,-13.6450,https://my.raceresult.com/103015/,Tías,NaN,Las Palmas
